In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
HERE = %pwd
sys.path.append(os.path.dirname(HERE))

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display

import numpy as np
import pandas as pd
import copy
import pickle
import time
import collections
from tqdm import tqdm
from collections import defaultdict

In [2]:
from src import utils
rng = utils.set_seed()

device = utils.device

In [3]:
# ---- Job / profile_mid-career ----
# User: Finance major, Law Clerk + Investment Intern + Consultant Intern
# Positive: Business Analyst (I1072362), Financial Analyst (I856888)

user_text = (
    "{'degree type': \"Bachelor's\", 'major': 'Finance', "
    "'graduation year': 2011, 'work history count': 3, "
    "'total years experience': 2, 'currently employed': 'No', "
    "'managed others': 'No', 'managed how many': 0, "
    "'work history': {'1': 'Law Clerk', "
    "'2': 'Investment Advisors Leadership Program Intern', "
    "'3': 'Associate Consultant Intern'}}"
)

d_documents = {
    "I1072362": (
        "title : Business Analyst for growing east-side company!\n"
        "description : An east-side distributor has an immediate need for a "
        "Business Analyst! Candidate will be responsible for producing sales "
        "& financial reports in support of growth strategies, and development "
        "of performance tracking tools vs. KPI targets. Ideal candidate MUST "
        "have advanced Excel & Access skills including pivot tables, Vlookups, "
        "conditional formatting, macros, financial modeling, and database development.\n"
        "requirements : +2 years experience in a business analyst role utilizing "
        "Advanced Excel & Access skills: pivot tables, Vlookups, macros, "
        "financial modeling and database development."
    ),
    "I856888": (
        "title : Financial Analyst\n"
        "description : A well established company in the Tri-Valley is looking "
        "for a Microsoft AX specialist to help with an implementation project. "
        "In this role you will help the Director of IT learn and work through "
        "Microsoft AX. Our client is most concerned with learning Procurement "
        "and Sales Order entry.\n"
        "requirements : Intermediate Order Entry, Intermediate Sales Analysis "
        "& Reporting, Advanced Procurement, Advanced Microsoft AX."
    ),
    "I562789": (
        "title : Front Desk Receptionist\n"
        "description : Downtown company is looking for a front desk receptionist "
        "to manage an office of 50 employees. Pay is $13 per hour plus benefits. "
        "Hours are M-F 8:00am-5:30pm. Must have 3+ years experience working a "
        "busy receptionist area plus have advanced MS office skills.\n"
        "requirements : Advanced skills in MS Office. Degree is a plus. "
        "3+ years experience working in a busy office."
    ),
    "I1100480": (
        "title : General Office Assistant\n"
        "description : Aerospace and Repair company looking to fill a General "
        "Office position. Responsibilities: Data Entry, Manage Outlook Calendars "
        "and Tasks, Answer Light Phones, Assist Sales Team, Update User Database.\n"
        "requirements : Preferable background in Medical Reception or Insurance "
        "Claim. Strong multi-tasking and organizational skills. "
        "Strong knowledge of Microsoft Office."
    ),
    "I602498": (
        "title : Customer Service Representative\n"
        "description : Established Los Angeles based Mailbox and Locker "
        "Manufacturing Company is seeking multiple Customer Service Representatives. "
        "Successful candidates must be bi-lingual, articulate, enthusiastic, "
        "highly motivated and possess excellent telephone skills.\n"
        "requirements : A minimum of 3 years Customer Service experience required."
    ),
}

s_flag = pd.Series({
    "I1072362": 1,
    "I856888": 1,
    "I562789": 0,
    "I1100480": 0,
    "I602498": 0,
})

df_candidates = pd.DataFrame({
    "item_text": pd.Series({item: d_documents[item] for item in s_flag.index}),
    "flag": s_flag
})

print("--" * 10, "user text", "--" * 10)
print(user_text)
print()
print("--" * 10, "item text", "--" * 10)
print(df_candidates["item_text"].iloc[0])

-------------------- user text --------------------
{'degree type': "Bachelor's", 'major': 'Finance', 'graduation year': 2011, 'work history count': 3, 'total years experience': 2, 'currently employed': 'No', 'managed others': 'No', 'managed how many': 0, 'work history': {'1': 'Law Clerk', '2': 'Investment Advisors Leadership Program Intern', '3': 'Associate Consultant Intern'}}

-------------------- item text --------------------
title : Business Analyst for growing east-side company!
description : An east-side distributor has an immediate need for a Business Analyst! Candidate will be responsible for producing sales & financial reports in support of growth strategies, and development of performance tracking tools vs. KPI targets. Ideal candidate MUST have advanced Excel & Access skills including pivot tables, Vlookups, conditional formatting, macros, financial modeling, and database development.
requirements : +2 years experience in a business analyst role utilizing Advanced Excel & 

## Embedding

In [4]:
# load embedding model
from sentence_transformers import SentenceTransformer
model_name_emb = "Qwen/Qwen3-Embedding-0.6B"
emb = SentenceTransformer(model_name_emb, device=device, trust_remote_code=True)

# embed user text (query) and candidate items (document)
v_user = emb.encode(user_text, prompt_name="query")
V_candidates = pd.DataFrame({
    item: emb.encode(item_text)
    for item, item_text in df_candidates["item_text"].to_dict().items()
}).T

# compute similarity
from sklearn.metrics.pairwise import cosine_similarity
s_sim = pd.Series(
    cosine_similarity([v_user], V_candidates.values)[0],
    index=V_candidates.index
)

# add similarity and sort
df_ = df_candidates.copy()
df_ = pd.concat([df_, pd.DataFrame({"score": s_sim})], axis=1)
df_ = df_.sort_values(by="score", ascending=False)
display(df_)

Loading weights: 100%|███████████████████████████████████████████| 310/310 [00:00<00:00, 567.44it/s, Materializing param=norm.weight]


,item_text,flag,score
I1072362,title : Business Analyst for growing east-side...,1,0.372152
I562789,title : Front Desk Receptionist\ndescription :...,0,0.357719
I856888,title : Financial Analyst\ndescription : A wel...,1,0.357554
I1100480,title : General Office Assistant\ndescription ...,0,0.296419
I602498,title : Customer Service Representative\ndescr...,0,0.248704


## LLM reranker

In [5]:
candidate_size = 5
at_K = 3
from src.reranker_llm import LLMReranker
llmreranker = LLMReranker(candidate_size=candidate_size)

items, flags, d_candidates = llmreranker._prepare(s_flag, d_documents)
prompt = llmreranker._prompt(user_text, d_candidates)

print("--" * 10, "prompt", "--" * 10)
print(prompt)

-------------------- prompt --------------------
# Task
Your task is to recommend exactly 10 items from the provided candidate set, ordered from most to least likely to be preferred by the user.

# Constraints
- Select items only from the provided candidate set; do not invent new items or IDs.
- Do not include any items the user has already interacted with.
- Return only a Python list literal of exactly 10 unique integer item IDs (the keys of the candidate set), ordered by preference, e.g., [8, 4, ...]. Do not output anything else.
- Make the ranking deterministic; if items are equally relevant, break ties by ascending item ID.
- Base your ranking only on the information in this prompt (user history and candidate metadata).

# Data
User Information:
{'degree type': "Bachelor's", 'major': 'Finance', 'graduation year': 2011, 'work history count': 3, 'total years experience': 2, 'currently employed': 'No', 'managed others': 'No', 'managed how many': 0, 'work history': {'1': 'Law Clerk', '

In [6]:
# call LLM (OpenAI API, GPT-4.1-mini)
import os
from openai import OpenAI
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

model_name_llm = "gpt-4.1-mini-2025-04-14"
response = client.chat.completions.create(
    model=model_name_llm,
    messages=[
        {"role": "system", "content": "You are an AI assistant that helps people find information."},
        {"role": "user", "content": prompt}
    ],
    temperature=0
)
output = response.choices[0].message.content
print(output)

[1, 5, 2, 3, 4]


In [7]:
df_score = llmreranker._add_score(output, items, flags)
display(df_score.sort_values(by="score", ascending=False))

,score,flag
I1072362,1.000000,1
I856888,0.500000,1
I1100480,0.333333,0
I562789,0.250000,0
I602498,0.200000,0


In [8]:
# call LLM (OpenAI API, GPT-5.4)

model_name_llm = "gpt-5.4-2026-03-05"

import os
from openai import OpenAI
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

response = client.responses.create(
    model=model_name_llm,
    input=[
        {"role": "user", "content": prompt}
    ],
    reasoning={"effort": "none"},
    temperature=0
)
output = response.output[-1].content[0].text
print(output)

[5, 1, 2, 3, 4]


In [9]:
df_score = llmreranker._add_score(output, items, flags)
display(df_score.sort_values(by="score", ascending=False))

,score,flag
I856888,1.000000,1
I1072362,0.500000,1
I1100480,0.333333,0
I562789,0.250000,0
I602498,0.200000,0


In [10]:
# NER: same logic as src/data_loader.py Loader.to_NER()
import spacy
nlp = spacy.load("en_core_web_sm")

def to_NER(text):
    doc = nlp(str(text))
    out, last = [], 0
    for ent in doc.ents:
        out.append(text[last:ent.start_char])
        out.append(f"<{ent.label_}>")
        last = ent.end_char
    out.append(text[last:])
    return "".join(out)

In [11]:
# ---- Job / profile_mid-career (NER=True) ----
print("--" * 10, "user text (NER)", "--" * 10)
print(to_NER(user_text))
print()
print("--" * 10, "item text (NER)", "--" * 10)
print(to_NER(df_candidates["item_text"].iloc[0]))

-------------------- user text (NER) --------------------
{'degree type': "Bachelor's", 'major': 'Finance', 'graduation year': <DATE>, 'work history count': <CARDINAL>, 'total <DATE> experience': <CARDINAL>, 'currently employed': 'No', 'managed others': 'No', 'managed how many': <CARDINAL>, 'work history': {<DATE>: '<PERSON>, '2': 'Investment Advisors Leadership Program Intern', '3': 'Associate Consultant Intern'}}

-------------------- item text (NER) --------------------
title : Business Analyst for growing east-side company!
description : An east-side distributor has an immediate need for a Business Analyst! Candidate will be responsible for producing sales & financial reports in support of growth strategies, and development of performance tracking tools vs. <ORG> targets. Ideal candidate <PERSON> have advanced <ORG> skills including pivot tables, <ORG>, conditional formatting, macros, financial modeling, and database development.
requirements : +2 <DATE> experience in a business an

In [12]:
# ---- MovieLens / profile ----
# User: Male, 35-44, technician/engineer
# Positive: drama classics (I954, I1104)

_ml_user_text = "{'gender': 'M', 'age': '35-44', 'occupation': 'technician/engineer'}"

_ml_d_documents = {
    "I954": "title : Mr. Smith Goes to Washington (1939)\ngenres : drama",
    "I1104": "title : Streetcar Named Desire, A (1951)\ngenres : drama",
    "I3409": "title : Final Destination (2000)\ngenres : drama, thriller",
    "I3157": "title : Stuart Little (1999)\ngenres : children's, comedy",
    "I1096": "title : Sophie's Choice (1982)\ngenres : drama",
}

_ml_s_flag = pd.Series({
    "I954": 1,
    "I1104": 1,
    "I3409": 0,
    "I3157": 0,
    "I1096": 0,
})

In [13]:
for f in [False, True]:
    print("==" * 20, f"NER flag: {f}", "==" * 20)
    _user = to_NER(_ml_user_text) if f else _ml_user_text
    _docs = {k: to_NER(v) for k, v in _ml_d_documents.items()} if f else _ml_d_documents
    _df = pd.DataFrame({
        "item_text": pd.Series({item: _docs[item] for item in _ml_s_flag.index}),
        "flag": _ml_s_flag
    })
    print("--" * 10, "user text", "--" * 10)
    print(_user)
    print()
    print("--" * 10, "item text", "--" * 10)
    print(_df["item_text"].iloc[0])

======================================== NER flag: False ========================================
-------------------- user text --------------------
{'gender': 'M', 'age': '35-44', 'occupation': 'technician/engineer'}

-------------------- item text --------------------
title : Mr. Smith Goes to Washington (1939)
genres : drama
======================================== NER flag: True ========================================
-------------------- user text --------------------
{'gender': 'M', 'age': '<DATE>, 'occupation': 'technician/engineer'}

-------------------- item text --------------------
title : Mr. <PERSON> to <GPE> (1939)
genres : drama


In [14]:
# ---- ARD_CDs_and_Vinyl / concat_3-sample ----
# User: New Age / Reiki meditation music listener
# Positive: meditation albums (I_B00UIUKYNU, I_B010U62XMQ)

_ard_history = [
    (
        "title : Reiki Hands of Love\n"
        "category : new age, meditation\n"
        "description : As human beings, we have endless possibilities to share. "
        "One of those possibilities is the healing and caring that can be expressed "
        "through our hands. The magic of love, the power of love, and the healing "
        "energy of love expressed in a touch."
    ),
    (
        "title : Reiki Healing Energy\n"
        "category : new age, meditation\n"
        "description : Reiki Healing Energy has the sweet flute along with the "
        "sounds of the sea birds and rolling tides lead to the mystical depths. "
        "Reiki comes from a Japanese word meaning universal life force energy. "
        "Terry Oldfield is a world-renowned artist whose profoundly inspiring "
        "music has touched the hearts of many."
    ),
    (
        "title : Koyasan: Reiki Sound Healing\n"
        "category : new age, healing\n"
        "description : Deuter puts his entire palette of musical abilities on display. "
        "This CD incorporates a global mixture of instruments, including the Chinese "
        "erhu, Japanese shakuhachi flute, East Indian tamboura and Tibetan bowls. "
        "Deuter takes the listener on a serene, mystical voyage through a variety "
        "of peaceful soundscapes."
    ),
]
_ard_user_text = "\n".join([f"#log {i+1}\n{t}" for i, t in enumerate(_ard_history)])

_ard_d_documents = {
    "I_B00UIUKYNU": (
        "title : Mystic Voyage\n"
        "category : new age, meditation\n"
        "description : Mystic Voyage contains various tracks from Deuter's ample "
        "array of healing music, specially selected for all healing practices or "
        "anyone wishing to have a taste of the beyond. This masterful compilation "
        "conveys the healing power of sound and silence."
    ),
    "I_B010U62XMQ": (
        "title : Illumination of the Heart\n"
        "category : new age, meditation\n"
        "description : Each track leads to the next in a crescendo of musical "
        "landscape that brings a sense of blissful serenity. Deuter plays an "
        "ample array of instruments including the flute, keyboard, cello, piano "
        "and guitar. Deuter is a master creator of spiritual music."
    ),
    "I_B01KN6XDS6": (
        "title : You Want It Darker\n"
        "category : pop\n"
        "description : 2016 release, the 14th studio album from the veteran "
        "singer/songwriter Leonard Cohen, the acclaimed composer of 'Hallelujah'."
    ),
    "I_B00D7JGL36": (
        "title : Bakersfield\n"
        "category : country, classic country\n"
        "description : MCA Recording artist Vince Gill and famed steel guitarist "
        "Paul Franklin have fashioned the album Bakersfield, a perfectly matched "
        "set of five Owens and five Haggard classics."
    ),
    "I_B00080EVB6": (
        "title : The Definitive Collection\n"
        "category : christian & gospel, country & bluegrass\n"
        "description : The most comprehensive full-length retrospective of one of "
        "the most acclaimed country bands, including their biggest hits: "
        "'Elizabeth,' 'My Only Love,' and 'Flowers on the Wall.'"
    ),
}

_ard_s_flag = pd.Series({
    "I_B00UIUKYNU": 1,
    "I_B010U62XMQ": 1,
    "I_B01KN6XDS6": 0,
    "I_B00D7JGL36": 0,
    "I_B00080EVB6": 0,
})

In [15]:
for f in [False, True]:
    print("==" * 20, f"NER flag: {f}", "==" * 20)
    _user = to_NER(_ard_user_text) if f else _ard_user_text
    _docs = {k: to_NER(v) for k, v in _ard_d_documents.items()} if f else _ard_d_documents
    _df = pd.DataFrame({
        "item_text": pd.Series({item: _docs[item] for item in _ard_s_flag.index}),
        "flag": _ard_s_flag
    })
    print("--" * 10, "user text", "--" * 10)
    print(_user)
    print()
    print("--" * 10, "item text", "--" * 10)
    print(_df["item_text"].iloc[0])

======================================== NER flag: False ========================================
-------------------- user text --------------------
#log 1
title : Reiki Hands of Love
category : new age, meditation
description : As human beings, we have endless possibilities to share. One of those possibilities is the healing and caring that can be expressed through our hands. The magic of love, the power of love, and the healing energy of love expressed in a touch.
#log 2
title : Reiki Healing Energy
category : new age, meditation
description : Reiki Healing Energy has the sweet flute along with the sounds of the sea birds and rolling tides lead to the mystical depths. Reiki comes from a Japanese word meaning universal life force energy. Terry Oldfield is a world-renowned artist whose profoundly inspiring music has touched the hearts of many.
#log 3
title : Koyasan: Reiki Sound Healing
category : new age, healing
description : Deuter puts his entire palette of musical abilities on di